### 프롬프트 템플릿  Agent tool rag에서 표준
- role:LLM이 어떤 역활을 할지 정함
- instruction:답변 규칙과 형식을 정리
- context:데이터베이스 검색결과처럼 답변에 참고할 실제 정보
- query:실제 질문


In [1]:
# 사용자 질문 -> LLM개입해서 사용자 의도를 파악해서
# 적절한 tool 호출 -> 라우터( Fast API )
# 수집한 정보를 -> LLM 전달
# 최종답변은 LLM

In [2]:
import os
from dotenv import load_dotenv
from openai import OpenAI
load_dotenv(override=True)

api_key = os.getenv('OPENAI_API_KEY')
if not api_key:
    raise EnvironmentError('openai api key .....')

class OpenAILLM:
    def __init__(self,model:str = 'GPT-5.4 nano'):
        self.client = OpenAI(api_key=api_key)
        self.model = model
    def generate(self, prompt:str) -> str:
        response = self.client.chat.completions.create(
            model = self.model,
            messages =[
                {"role":"system", "content":"You are an ecomerce recommendation assistant, Return only valid JSON"},
                {"role":"user", "content":prompt}
            ],
            temperature=0,
            response_format={'type':'json_object'}
        )
        return response.choices[0].message.content
llm = OpenAILLM()
llm.model  

'GPT-5.4 nano'

In [6]:
# 데이터
do_search_results = [
    {"id":"p1", "name":"고성능 노트북","category":"가전"},
    {"id":"p2", "name":"사무용 랩탑","category":"가전"},
    {"id":"p3", "name":"미러리스 카메라","category":"가전"},
]
context_string = ""
for item in do_search_results:
    context_string += f"- 상품명:{item['name']} | 카테고리:{item['category']}"

In [7]:
# 프롬프트 템플릿 - 고정된 구조
from  textwrap import dedent  #  들여쓰기 indent를 제거
user_query = '코딩할 때 쓸만한 노트북 하나 추천해줘, 믿을만한 제품으로'
raw_prompt = dedent(f'''
                    당신은 이커머스 전문 AI 추천 어시스턴트입니다.
                    아래 컨텍스트를 참고하여 사용자의 질문에 답하세요
                    반드시 json 형식으로만 답하세요

                    [컨텍스트]
                    {context_string}

                    [질문]
                    {user_query}

                    답변은 반드시 아래 형식의 json으로만 출력하세요
                    {
                        {
                            "recommended_product" : "상품명",
                            "reason":"추천사유"                            
                        }
                    }
                    ''')
print(raw_prompt)


당신은 이커머스 전문 AI 추천 어시스턴트입니다.
아래 컨텍스트를 참고하여 사용자의 질문에 답하세요
반드시 json 형식으로만 답하세요

[컨텍스트]
- 상품명:고성능 노트북 | 카테고리:가전- 상품명:사무용 랩탑 | 카테고리:가전- 상품명:미러리스 카메라 | 카테고리:가전

[질문]
코딩할 때 쓸만한 노트북 하나 추천해줘, 믿을만한 제품으로

답변은 반드시 아래 형식의 json으로만 출력하세요
{'recommended_product': '상품명', 'reason': '추천사유'}

